# 06.7 - PyTorch Datasets & DataLoaders

**Phase:** 06 - Deep Learning

**Status:** VERIFIED

---

## 1. What Are We Solving?

`Dataset` and `DataLoader` handle data loading, batching, shuffling, and parallel preprocessing — the bridge between raw data and the training loop.

## 2. Why Does This Matter?

Real datasets don't fit in memory or come as clean tensors. Efficient data pipelines feed batches on demand; poor loading is an invisible bottleneck.

## 3. Prerequisites

- Unit 06.4 (PyTorch basics), basic file I/O

## 4. Learning Objectives

By the end of this unit, you should be able to:
- Build custom `Dataset`s and efficient `DataLoader`s
- Implement train/val/test splits with batching
- Identify and fix data-loading bottlenecks

## 5. Mental Model

A `Dataset` is a library card catalog — it knows where each data point lives. A `DataLoader` is the librarian — it grabs items in batches, shuffles the shelf, and hands them over efficiently.


## 6. Backend


In [1]:
import matplotlib
matplotlib.use('Agg')
import torch
from torch.utils.data import Dataset, DataLoader, TensorDataset, random_split
import torch.nn as nn
import torch.optim as optim
print("PyTorch version:", torch.__version__)


PyTorch version: 2.13.0+cpu


## 7. TensorDataset + DataLoader Basics


In [2]:
torch.manual_seed(42)
X = torch.randn(1000, 20)
y = torch.randint(0, 2, (1000,))
dataset = TensorDataset(X, y)
loader = DataLoader(dataset, batch_size=64, shuffle=True)

print(f"Dataset length: {len(dataset)}")
batch_X, batch_y = next(iter(loader))
print(f"Batch X shape: {batch_X.shape}, y shape: {batch_y.shape}")
print(f"Number of batches: {len(loader)}")


Dataset length: 1000


Batch X shape: torch.Size([64, 20]), y shape: torch.Size([64])
Number of batches: 16


## 8. Custom Dataset

Subclass `Dataset` with `__len__` and `__getitem__`. Here we build a synthetic 'image-like' dataset.


In [3]:
class SynthDataset(Dataset):
    def __init__(self, n, img_size=8, noise=0.1):
        self.images = torch.randn(n, 1, img_size, img_size) * noise
        # label = whether mean intensity > 0 (a learnable-ish rule)
        self.labels = (self.images.mean(dim=(1,2,3)) > 0).long()
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        return self.images[idx], self.labels[idx]

ds = SynthDataset(500)
print(f"Custom dataset length: {len(ds)}")
img, lab = ds[3]
print(f"Sample[3] image shape: {img.shape}, label: {lab.item()}")


Custom dataset length: 500
Sample[3] image shape: torch.Size([1, 8, 8]), label: 1


## 9. Train/Validation/Test Splits

Use `random_split` into three DataLoaders.


In [4]:
torch.manual_seed(7)
full = SynthDataset(1000)
n = len(full)
tr_ds, val_ds, te_ds = random_split(full, [int(0.6*n), int(0.2*n), n - int(0.6*n) - int(0.2*n)])

train_loader = DataLoader(tr_ds, batch_size=64, shuffle=True, drop_last=True)
val_loader   = DataLoader(val_ds, batch_size=64, shuffle=False)
test_loader  = DataLoader(te_ds, batch_size=64, shuffle=False)

print(f"Train/Val/Test sizes: {len(tr_ds)}/{len(val_ds)}/{len(te_ds)}")
print(f"Train loader batches: {len(train_loader)} (drop_last removes one incomplete batch)")
print(f"Val   loader batches: {len(val_loader)}")


Train/Val/Test sizes: 600/200/200
Train loader batches: 9 (drop_last removes one incomplete batch)
Val   loader batches: 4


## 10. Train a Tiny Model Through a DataLoader

Iterate over batches instead of passing the whole tensor.


In [5]:
model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(8*8, 32), nn.ReLU(),
    nn.Linear(32, 2)
)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

losses = []
for epoch in range(3):
    epoch_loss = 0.0
    for bx, by in train_loader:
        optimizer.zero_grad()
        loss = criterion(model(bx), by)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * len(bx)
    losses.append(epoch_loss / len(tr_ds))
    print(f"epoch {epoch}: avg loss = {losses[-1]:.4f}")

correct = 0; total = 0
model.eval()
with torch.no_grad():
    for bx, by in test_loader:
        correct += (model(bx).argmax(dim=1) == by).sum().item()
        total += len(by)
print(f"Test accuracy through DataLoader: {correct/total:.3f}")


epoch 0: avg loss = 0.6399
epoch 1: avg loss = 0.5330
epoch 2: avg loss = 0.3670
Test accuracy through DataLoader: 0.885


## 11. Debugging & Considerations

| Symptom | Possible Cause | Verify | Fix |
|---|---|---|---|
| Slow DataLoader | num_workers=0 | Check worker count | Set num_workers=2-4 |
| Low GPU util | Load bottleneck | Monitor GPU vs CPU | More workers, pin_memory |
| Memory during load | Loading all at once | Check __getitem__ | Load lazily |
| Inconsistent batch sizes | drop_last=False | Check last batch | drop_last=True for training |

## 12. Common Mistakes

- Forgetting `shuffle=True` for training (bad minibatches).
- No `drop_last` → last tiny batch distorts normalization.
- `__getitem__` doing heavy IO per call with no caching.

## 13. Real-World Considerations

- Medical imaging pipelines load DICOM, augment, normalize per sample via custom Dataset, and batch 32 with 4 workers.
- `pin_memory=True` speeds GPU copies.

## 14. When NOT to Use

- A custom Dataset isn't needed if data already fits as tensors — use `TensorDataset`.

## 15. Challenge

Add a simple geometric augmentation in `__getitem__` (random sign flip) and confirm the effective dataset grows.


In [6]:
# Challenge: augmentation inside __getitem__
class AugmentedDataset(Dataset):
    def __init__(self, n, img_size=8):
        self.images = torch.randn(n, 1, img_size, img_size) * 0.1
        self.labels = (self.images.mean(dim=(1,2,3)) > 0).long()
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        img = self.images[idx]
        if torch.rand(1).item() < 0.5:      # random horizontal flip
            img = torch.flip(img, dims=[2])
        return img, self.labels[idx]

ads = AugmentedDataset(200)
im0, _ = ads[0]
im1, _ = ads[0]
same = torch.equal(im0, im1)
print(f"Two calls to [0] identical? {same}  (False means augmentation fires)")
print("\nAugmentation on-the-fly multiplies effective training diversity.")


Two calls to [0] identical? False  (False means augmentation fires)

Augmentation on-the-fly multiplies effective training diversity.


## 16. Closed-Book Recall

Without looking back:

1. What does `shuffle=True` do and why is it important?
2. Why use `pin_memory=True` on GPU?
3. Difference between `Dataset` and `DataLoader`?
4. When would you use `drop_last=True`?

## 17. Teach-Back Questions

Explain to another person:

- The two methods a Dataset must implement.
- How DataLoader batches and shuffles.

## 18. Summary

You built TensorDatasets, custom Datasets, splits, and trained through DataLoaders; explored augmentation.

## 19. Further Experiment

- Time loading with num_workers=0 vs 2.
- Add `pin_memory=True` and compare.

## 20. Verification Status

```
STATUS: VERIFIED
EXECUTION: PASS
DEPENDENCIES: torch
OUTPUTS: PASS
LAST VERIFIED: 2026-08-29
```
